In [0]:
# Databricks notebook source
# MAGIC %md
# MAGIC # Automated Lakehouse Data Quality & Integrity Test Suite
# MAGIC 
# MAGIC - Validates data contracts across Bronze, Silver, and Gold layers.
# MAGIC - Asserts Primary/Surrogate Key uniqueness, SCD Type 2 date interval consistency, and referential integrity.



In [0]:
# COMMAND ----------

from pyspark.sql import functions as F

class LakehouseTestSuite:
    def __init__(self):
        self.passed_tests = 0
        self.failed_tests = []

    def assert_true(self, condition: bool, test_name: str, failure_msg: str):
        if condition:
            print(f"  [PASS] {test_name}")
            self.passed_tests += 1
        else:
            print(f"  [FAIL] {test_name} -> {failure_msg}")
            self.failed_tests.append((test_name, failure_msg))

    def run_all(self):
        print("Starting Lakehouse Test Execution...\n" + "="*50)
        
        # 1. Bronze Layer Tests
        self.test_bronze_layer()
        
        # 2. Silver Dimensions Tests
        self.test_silver_customer_scd2()
        self.test_silver_products()
        
        # 3. Silver Fact Tests
        self.test_silver_fact_orders()
        
        # 4. Gold Layer Tests
        self.test_gold_kpis()
        
        print("\n" + "="*50)
        print(f"Test Summary: {self.passed_tests} PASSED, {len(self.failed_tests)} FAILED")
        if self.failed_tests:
            raise AssertionError(f"Quality suite failed with {len(self.failed_tests)} errors: {self.failed_tests}")

    def test_bronze_layer(self):
        print("\n[Layer: 01_Bronze]")
        # Assert key bronze tables exist and have records
        bronze_tables = ["user_addr_scd2", "cart_items", "catalog_data"]
        for tbl in bronze_tables:
            if spark.catalog.tableExists(f"bronze.{tbl}"):
                cnt = spark.table(f"bronze.{tbl}").count()
                self.assert_true(cnt > 0, f"bronze.{tbl} row count check", f"Table has 0 records")
            else:
                self.assert_true(False, f"bronze.{tbl} exists", "Table not registered")

    def test_silver_customer_scd2(self):
        print("\n[Layer: 02_Silver - dim_customers_scd2]")
        df = spark.table("silver.dim_customers_scd2")
        
        # Check 1: customer_sk uniqueness
        total_rows = df.count()
        distinct_sk = df.select("customer_sk").distinct().count()
        self.assert_true(total_rows == distinct_sk, "Customer Surrogate Key Uniqueness", f"Found {total_rows - distinct_sk} duplicates")
        
        # Check 2: Valid Date Ranges (valid_to >= valid_from)
        invalid_dates = df.filter(F.col("valid_to") < F.col("valid_from")).count()
        self.assert_true(invalid_dates == 0, "SCD2 Interval Validity", f"Found {invalid_dates} rows where valid_to < valid_from")
        
        # Check 3: Current flag consistency (is_current has 9999-12-31)
        mismatched_current = df.filter((F.col("is_current") == True) & (F.col("valid_to") != F.to_date(F.lit("9999-12-31")))).count()
        self.assert_true(mismatched_current == 0, "is_current Flag Logic", f"Found {mismatched_current} inconsistent current records")

    def test_silver_products(self):
        print("\n[Layer: 02_Silver - dim_products_catalog]")
        df = spark.table("silver.dim_products_catalog")
        
        # Check 1: product_sk uniqueness
        total_rows = df.count()
        distinct_sk = df.select("product_sk").distinct().count()
        self.assert_true(total_rows == distinct_sk, "Product Surrogate Key Uniqueness", f"Found {total_rows - distinct_sk} duplicates")
        
        # Check 2: No negative prices
        neg_prices = df.filter(F.col("unit_price") < 0).count()
        self.assert_true(neg_prices == 0, "Non-Negative Unit Prices", f"Found {neg_prices} products with price < 0")

    def test_silver_fact_orders(self):
        print("\n[Layer: 02_Silver - fct_orders_fulfillment]")
        df = spark.table("silver.fct_orders_fulfillment")
        
        # Check 1: fact_order_sk uniqueness
        total_rows = df.count()
        distinct_sk = df.select("fact_order_sk").distinct().count()
        self.assert_true(total_rows == distinct_sk, "Fact Order Key Uniqueness", f"Found {total_rows - distinct_sk} duplicates")
        
        # Check 2: Non-null order identifier
        null_orders = df.filter(F.col("order_id").isNull()).count()
        self.assert_true(null_orders == 0, "Non-Null Order Identifiers", f"Found {null_orders} null order IDs")

    def test_gold_kpis(self):
        print("\n[Layer: 03_Gold]")
        df_rfm = spark.table("gold.customer_rfm_segments")
        df_kpis = spark.table("gold.daily_sales_fulfillment_kpis")
        
        # Check 1: Gold RFM has populated segments
        null_segments = df_rfm.filter(F.col("customer_segment").isNull()).count()
        self.assert_true(null_segments == 0, "RFM Customer Segment Completeness", f"Found {null_segments} unassigned customer segments")
        
        # Check 2: Gold daily KPIs table has positive aggregate counts
        kpi_count = df_kpis.count()
        self.assert_true(kpi_count > 0, "Daily Sales KPI Population", "gold.daily_sales_fulfillment_kpis has 0 records")

# COMMAND ----------


In [0]:

# Execute the test suite
runner = LakehouseTestSuite()
runner.run_all()